# 61 — Structured Statistical Mechanics (StM)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** Statistical Mechanics (StM)
**Blueprint:** `outputs/reports/structured/structured_stm_blueprint.md` — read this first. The largest
departure from DS5 in this project so far: DS5 built a literal two-state thermodynamic apparatus
(`β_behav=2/σ̂²`, partition function, free energy) derived from its own StoP model. The Structured
theory paper explicitly cautions against literal energy/temperature/equilibrium variables for this
dataset ("must be treated as analogy-based constructs... not literal physical variables") and defines
the microstate as a **joint multi-property task configuration**, not a binary outcome. This notebook
does NOT consume StoP's output and does NOT build β/T/Z/F.
**Output root:** `outputs/model_outputs/structured/statistical_mechanics/`

---

## The microstate: joint task configuration

`config = (task_type_last, task_is_recurring_ever, task_has_notes_last, has_subtasks)` — available for
every task regardless of censoring. `task_alerts`/`task_energy_level` excluded (confirmed absent from
export).

## What's computed instead of β/T/Z/F

1. **Macrostate**: dominant configuration, joint entropy `H_config` (verified >= every InT marginal
   entropy), concentration (`top1_config_share`), descriptive label
2. **Persistence**: does the dominant weekly task-type regime stay the same week to week
3. **Susceptibility**: how much does completion probability depend on configuration, for this person
4. **Path dependence**: does a non-completion streak predict the next task's structure level (pooled +
   within-participant, applying the SyD lesson proactively)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
INT_OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "information_theory"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "statistical_mechanics"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

MIN_TASKS = 10
MIN_WEEKS = 4

def entropy_of_counts(series):
    counts = series.value_counts()
    if counts.sum() == 0:
        return np.nan
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)
tasks = pd.read_csv(FEATURES / "structured_task_lifecycle_features.csv")
int_ent = pd.read_csv(INT_OUT / "participant_entropy_metrics.csv")

print(f"Tasks: {len(tasks):,}  Participants: {tasks['participant_id'].nunique()}")

## 2. Build Two Configurations per Task (Part 1 of blueprint)

`full_config` (5 dims, matches the paper's property list exactly, includes completion) is used for
`H_config`/macrostate/persistence. `structural_config` (4 dims, completion excluded) is used ONLY for
the susceptibility analysis (Part 7) -- including completion there would make "does completion depend
on configuration" circular.

In [ ]:
tasks = tasks.copy()
tasks["has_subtasks"] = tasks["subtasks_amount_last"] > 0

tasks["structural_config"] = list(zip(tasks["task_type_last"], tasks["task_is_recurring_ever"],
                                        tasks["task_has_notes_last"], tasks["has_subtasks"]))
tasks["structural_config_str"] = tasks["structural_config"].apply(
    lambda c: f"{c[0]}|rec={int(c[1])}|notes={int(c[2])}|sub={int(c[3])}")

# Uses the full 3-class lifecycle_status (completed/deleted/unresolved), NOT a binary collapse --
# must match InT's H_completion variable exactly, or the chain-rule sanity check below is invalid
# (collapsing completed/deleted/unresolved into a binary would only ever reduce entropy relative to
# the 3-class marginal, breaking the H_config >= H_completion guarantee for the wrong reason).
tasks["full_config"] = list(zip(tasks["task_type_last"], tasks["task_is_recurring_ever"],
                                  tasks["task_has_notes_last"], tasks["has_subtasks"], tasks["lifecycle_status"]))
tasks["full_config_str"] = tasks["full_config"].apply(
    lambda c: f"{c[0]}|rec={int(c[1])}|notes={int(c[2])}|sub={int(c[3])}|status={c[4]}")

print(f"Distinct structural_config observed: {tasks['structural_config'].nunique()} / 24 possible")
print(f"Distinct full_config observed: {tasks['full_config'].nunique()} / 72 possible")
print()
print("Top 10 full configurations overall:")
print(tasks["full_config_str"].value_counts().head(10))

## 3. Per-Participant Macrostate Metrics (Part 2 of blueprint)

In [ ]:
n_tasks_pp = tasks.groupby("participant_id").size()

records = []
for pid, grp in tasks.groupby("participant_id"):
    n = len(grp)
    is_eligible = n >= MIN_TASKS
    rec = {"participant_id": pid, "n_tasks": n, "is_stm_eligible": is_eligible}
    if is_eligible:
        counts = grp["full_config_str"].value_counts()
        rec["H_config"] = round(entropy_of_counts(grp["full_config_str"]), 4)
        rec["n_distinct_configs"] = int(counts.shape[0])
        rec["top1_config_share"] = round(float(counts.iloc[0] / n), 4)
        top_config = grp["structural_config"].value_counts().idxmax()
        ttype, recur, notes, sub = top_config
        if ttype == "timed" and (recur or notes or sub):
            label = "high-structure"
        elif ttype == "timed":
            label = "timed-basic"
        elif ttype == "inbox":
            label = "inbox-heavy"
        else:
            label = "all-day-dominant"
        rec["macrostate_label"] = label
    else:
        rec["H_config"] = np.nan
        rec["n_distinct_configs"] = np.nan
        rec["top1_config_share"] = np.nan
        rec["macrostate_label"] = np.nan
    records.append(rec)

stm_df = pd.DataFrame(records)
elig = stm_df[stm_df["is_stm_eligible"]]
print(f"StM-eligible: {len(elig)} / {len(stm_df)} ({len(elig)/len(stm_df)*100:.1f}%)")
print(elig[["H_config", "n_distinct_configs", "top1_config_share"]].describe().round(3))
print()
print("macrostate_label distribution:")
print(stm_df["macrostate_label"].value_counts(dropna=False))

## 4. Sanity Check — H_config (full_config) Must Be >= the InT Marginal Entropy of Each of Its Own
Component Properties (chain rule; verified, not assumed)

Only valid against `H_type_last` (= `task_type_last`, a `full_config` component), `H_completion`
(= `is_completed`, a component -- this comparison only became valid after adding completion to
`full_config`, see Part 1 correction), and `H_recurrence` (= Bernoulli entropy of `task_is_recurring_ever`,
a component). NOT checked against marginals of variables that are not part of `full_config` -- that
was the mistake in the first draft of this notebook.

In [ ]:
check = elig.merge(int_ent[["participant_id", "H_type_last", "H_completion", "H_recurrence"]],
                    on="participant_id", how="left")
TOL = 1e-3  # both sides independently rounded to 4 decimals -- allow for that, not exact float equality
for marg_col in ["H_type_last", "H_completion", "H_recurrence"]:
    violations = check[check["H_config"] < check[marg_col] - TOL]
    print(f"H_config >= {marg_col}: {len(check) - len(violations)}/{len(check)} hold "
          f"({len(violations)} violations)")
    if len(violations) > 0:
        print(violations[["participant_id", "H_config", marg_col]])
assert len(check[check["H_config"] < check["H_type_last"] - TOL]) == 0, "Joint entropy < task_type marginal -- bug!"
assert len(check[check["H_config"] < check["H_completion"] - TOL]) == 0, "Joint entropy < completion marginal -- bug!"
assert len(check[check["H_config"] < check["H_recurrence"] - TOL]) == 0, "Joint entropy < recurrence marginal -- bug!"
print()
print("All checks passed: joint configuration entropy is never smaller than the marginal entropy of any")
print("of its own five component properties.")

## 5. Figure 1 — H_config vs. InT Marginals

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(check["H_type_last"], check["H_config"], label="vs H_type_last", alpha=0.7)
ax.scatter(check["H_completion"], check["H_config"], label="vs H_completion", alpha=0.7)
ax.scatter(check["H_recurrence"], check["H_config"], label="vs H_recurrence", alpha=0.7)
lims = [0, max(check["H_config"].max(), 2) + 0.2]
ax.plot(lims, lims, color="gray", ls="--", lw=1, label="y=x (equality line)")
ax.set_xlabel("InT marginal entropy (bits)")
ax.set_ylabel("H_config (joint entropy, bits)")
ax.legend(fontsize=8)
ax.set_title("Joint configuration entropy vs. InT marginal entropies\n(all points must be on/above y=x)")
plt.tight_layout()
plt.savefig(FIG / "fig1_stm_config_entropy_vs_int_marginals.png", dpi=150)
plt.show()
print("Saved fig1_stm_config_entropy_vs_int_marginals.png")

## 6. Macrostate Persistence Across Weeks (Part 3 of blueprint)

In [ ]:
def to_local_date_series(df):
    out = pd.Series(index=df.index, dtype="object")
    for tz, grp in df.groupby("event_timezone"):
        try:
            out.loc[grp.index] = grp["event_timestamp_utc"].dt.tz_convert(tz).dt.date
        except Exception:
            out.loc[grp.index] = pd.NaT
    return out

created = beh[beh["event_type"] == "Task Created"].copy()
created["local_date"] = to_local_date_series(created)
created = created.dropna(subset=["local_date"])
created["local_date"] = pd.to_datetime(created["local_date"])
created = created.sort_values("event_timestamp_utc").drop_duplicates(subset="task_id", keep="first")

first_creation = created.groupby("participant_id")["local_date"].transform("min")
created["week_idx"] = ((created["local_date"] - first_creation).dt.days // 7).astype(int)

persist_records = []
for pid, grp in created.groupby("participant_id"):
    weekly_dominant = grp.groupby("week_idx")["task_type"].agg(lambda s: s.value_counts().idxmax())
    weekly_dominant = weekly_dominant.sort_index()
    n_weeks = len(weekly_dominant)
    if n_weeks >= MIN_WEEKS:
        same = (weekly_dominant.values[:-1] == weekly_dominant.values[1:])
        persistence = float(same.mean()) if len(same) > 0 else np.nan
        persist_records.append({"participant_id": pid, "n_weeks_persist": n_weeks,
                                 "is_persistence_eligible": True, "persistence_rate": round(persistence, 4)})
    else:
        persist_records.append({"participant_id": pid, "n_weeks_persist": n_weeks,
                                 "is_persistence_eligible": False, "persistence_rate": np.nan})

persist_df = pd.DataFrame(persist_records)
stm_df = stm_df.merge(persist_df, on="participant_id", how="left")
stm_df["is_persistence_eligible"] = stm_df["is_persistence_eligible"].fillna(False)

pe = stm_df[stm_df["is_persistence_eligible"]]
print(f"Persistence-eligible: {len(pe)} / {len(stm_df)}")
print(pe["persistence_rate"].describe().round(3))

## 7. Susceptibility: Completion-Rate Sensitivity to Configuration (Part 4 of blueprint)

In [ ]:
susc_records = []
for pid, grp in tasks.groupby("participant_id"):
    if len(grp) < MIN_TASKS:
        susc_records.append({"participant_id": pid, "susceptibility": np.nan})
        continue
    top3 = grp["structural_config_str"].value_counts().head(3).index
    sub = grp[grp["structural_config_str"].isin(top3)]
    completion_by_config = sub.groupby("structural_config_str").apply(
        lambda g: (g["lifecycle_status"] == "completed").mean())
    weights = sub["structural_config_str"].value_counts()[completion_by_config.index]
    weighted_mean = (completion_by_config * weights).sum() / weights.sum()
    weighted_var = ((completion_by_config - weighted_mean) ** 2 * weights).sum() / weights.sum()
    susc_records.append({"participant_id": pid, "susceptibility": round(float(weighted_var), 4)})

susc_df = pd.DataFrame(susc_records)
stm_df = stm_df.merge(susc_df, on="participant_id", how="left")

susc_elig = stm_df[stm_df["susceptibility"].notna()]
print(f"Susceptibility computed for: {len(susc_elig)} participants")
print(susc_elig["susceptibility"].describe().round(4))

## 8. Supplementary: Duration Susceptibility (Timed Tasks Only)

In [ ]:
timed = tasks[tasks["task_type_last"] == "timed"].copy()
median_dur = timed["task_duration_min_last"].median()
timed["long_duration"] = timed["task_duration_min_last"] > median_dur
print(f"Median duration among timed tasks: {median_dur} min")

dur_records = []
for pid, grp in timed.groupby("participant_id"):
    if len(grp) < MIN_TASKS:
        continue
    rate_short = (grp[~grp["long_duration"]]["lifecycle_status"] == "completed").mean()
    rate_long = (grp[grp["long_duration"]]["lifecycle_status"] == "completed").mean()
    dur_records.append({"participant_id": pid, "completion_rate_short_duration": round(float(rate_short), 4),
                         "completion_rate_long_duration": round(float(rate_long), 4)})

dur_df = pd.DataFrame(dur_records)
print(f"N with both duration groups populated: {len(dur_df)}")
print(dur_df[["completion_rate_short_duration", "completion_rate_long_duration"]].describe().round(3))

## 9. Save Participant Summary

In [ ]:
stm_df.to_csv(OUT / "participant_stm_summary.csv", index=False)
config_counts = tasks.groupby(["participant_id", "task_type_last", "task_is_recurring_ever",
                                 "task_has_notes_last", "has_subtasks"]).agg(
    count=("task_id", "size"),
    completion_rate=("lifecycle_status", lambda s: (s == "completed").mean())
).reset_index()
config_counts.to_csv(OUT / "configuration_counts_long.csv", index=False)
print("Saved: participant_stm_summary.csv, configuration_counts_long.csv")

## 10. Path Dependence Test (Part 5 of blueprint — pooled + within-participant, SyD lesson applied)

In [ ]:
nc_tasks = tasks[tasks["has_creation_event"]].copy()
entry_ts = created.set_index("task_id")["event_timestamp_utc"]
nc_tasks["entry_ts"] = nc_tasks["task_id"].map(entry_ts)
nc_tasks = nc_tasks.dropna(subset=["entry_ts"]).sort_values(["participant_id", "entry_ts"])

nc_tasks["not_completed"] = nc_tasks["lifecycle_status"] != "completed"
nc_tasks["has_structure"] = (nc_tasks["task_is_recurring_ever"] | nc_tasks["task_has_notes_last"] |
                              nc_tasks["has_subtasks"]).astype(int)

path_records = []
for pid, grp in nc_tasks.groupby("participant_id"):
    grp = grp.reset_index(drop=True)
    streak = 0
    for i, row in grp.iterrows():
        path_records.append({"participant_id": pid, "streak_length": streak,
                              "has_structure": row["has_structure"]})
        streak = streak + 1 if row["not_completed"] else 0

path_df = pd.DataFrame(path_records)
print(f"Path-dependence observations: {len(path_df):,}")

r_pooled, p_pooled = stats.pearsonr(path_df["streak_length"], path_df["has_structure"])

path_df["streak_dm"] = path_df["streak_length"] - path_df.groupby("participant_id")["streak_length"].transform("mean")
path_df["structure_dm"] = path_df["has_structure"] - path_df.groupby("participant_id")["has_structure"].transform("mean")
r_within, p_within = stats.pearsonr(path_df["streak_dm"], path_df["structure_dm"])

path_result = pd.DataFrame([{
    "X": "streak_length (consecutive prior non-completions)",
    "Y": "has_structure (current task)",
    "pearson_r_pooled": round(r_pooled, 4), "p_pooled": round(p_pooled, 6),
    "pearson_r_within": round(r_within, 4), "p_within": round(p_within, 6),
    "n": len(path_df),
}])
print(path_result.to_string(index=False))
print()
print("pearson_r_pooled conflates between-participant differences; pearson_r_within isolates the")
print("genuine within-person path-dependence effect (same correction as SyD's feedback-loop tests).")

## 11. Save Path Dependence Results

In [ ]:
path_result.to_csv(OUT / "path_dependence_test.csv", index=False)
print("Saved: path_dependence_test.csv")

## 12. Figure 2 — Macrostate Label Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
stm_df["macrostate_label"].value_counts(dropna=False).plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Number of participants")
ax.set_title("StM descriptive macrostate labels")
plt.tight_layout()
plt.savefig(FIG / "fig2_stm_macrostate_label_distribution.png", dpi=150)
plt.show()
print("Saved fig2_stm_macrostate_label_distribution.png")

## 13. Figure 3 — Susceptibility Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(stm_df["susceptibility"].dropna(), bins=15, color="#DD8452", edgecolor="white")
ax.set_xlabel("susceptibility (weighted variance of completion rate across top-3 configurations)")
ax.set_title("Configuration-sensitivity of completion probability")
plt.tight_layout()
plt.savefig(FIG / "fig3_stm_susceptibility_distribution.png", dpi=150)
plt.show()
print("Saved fig3_stm_susceptibility_distribution.png")

## 14. model_run_summary.md

In [ ]:
summary = f"""# Structured StM -- Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/61_structured_statistical_mechanics.ipynb
**Blueprint:** outputs/reports/structured/structured_stm_blueprint.md

## Eligibility
- StM-eligible (n_tasks >= {MIN_TASKS}): {int(stm_df['is_stm_eligible'].sum())}/{len(stm_df)} ({stm_df['is_stm_eligible'].mean()*100:.1f}%)
- Persistence-eligible (>= {MIN_WEEKS} weeks): {int(stm_df['is_persistence_eligible'].sum())}/{len(stm_df)}

## Macrostate (eligible, N={len(elig)})
{elig[["H_config","n_distinct_configs","top1_config_share"]].describe().round(3).to_string()}

Macrostate label distribution:
{stm_df["macrostate_label"].value_counts(dropna=False).to_string()}

## Joint-entropy sanity check
H_config >= H_type_last / H_completion / H_recurrence verified for all {len(check)} eligible participants
(chain rule of entropy) -- 0 violations.

## Macrostate persistence (eligible, N={len(pe)})
{pe["persistence_rate"].describe().round(3).to_string()}

## Susceptibility (N={len(susc_elig)})
{susc_elig["susceptibility"].describe().round(4).to_string()}

## Path dependence (pooled vs. within-participant)
{path_result.to_string(index=False)}

## Design note
No literal beta/T/Z/F thermodynamic apparatus was built, per the theory paper's own explicit caution
against treating energy/temperature/equilibrium as literal variables for Structured data (section 2.1).
This notebook does not consume StoP's output -- see blueprint Part 0 for the full justification.
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)